# Breaker-and-a-Half Substation

The breaker-and-a-half configuration has two main buses joined by diameters. Each diameter has three
breakers in series between bus 1 and bus 2, with a branch connected on each side of the middle breaker, so
each branch has one and a half breakers. Any breaker, or either bus, can be taken out of service without
interrupting a branch, which makes it common in high-voltage transmission substations.

![Breaker-and-a-Half Substation](../images/breaker_and_half.png)

## CIM representation

The two buses are nodes with `BusbarSection`s. Each diameter is three breaker bays in series, through two
side nodes: bus 1 to side 1, side 1 to side 2, and side 2 to bus 2. A branch connects to one side node
through one `Disconnector`.

| Object | mRIDs |
|---|---|
| Bus 1, bus 2 | nodes `vl_115_bus_1` and `vl_115_bus_2`, BusbarSections `vl_115_bus_1_bbs` and `vl_115_bus_2_bbs` |
| Diameter *d*, side nodes | `vl_115_dia_d_side_1`, `vl_115_dia_d_side_2` |
| Diameter *d*, breaker bays | `vl_115_dia_d_1` from bus 1 to side 1, `vl_115_dia_d_2` from side 1 to side 2, `vl_115_dia_d_3` from side 2 to bus 2 |
| Branch on diameter *d*, side *s* | Disconnector `vl_115_dia_d_side_s_d`, from the side node to the branch node |

A *breaker bay* `p` is the chain `p_d1` - `p_j1` - `p_brk` - `p_j2` - `p_d2`: a `Breaker` between two
`Disconnector`s, with a junction node on either side of the breaker. See
[Substation builder](2_1_substation_builder.ipynb) for the naming rules and the call order shared by every
layout.

The diagram comes from an earlier release of CIM-Builder, and its labels are not the mRIDs written now.
The table gives the current mRIDs, all prefixed with the VoltageLevel mRID `vl_115`.

## Build the substation

### Set up

The CIM profile and unit handling are process-wide environment variables, so set them before importing
CIMantic Graphs.

In [1]:
import logging
import os

os.environ["CIMG_CIM_PROFILE"] = "cim18v17"
os.environ["CIMG_USE_UNITS"] = "TRUE"
os.environ["CIMG_SYSTEM_MVA_BASE"] = "100"
# The IEEE 13 files carry CIMHub extensions that cim18v17 does not have: hide the parser warnings
logging.getLogger("cimgraph").setLevel(logging.ERROR)

import cimgraph.utils as utils
import cimprofiles.cim18.cim18v17 as cim
from cimgraph.databases import XMLFile
from cimgraph.models import FeederModel
from cimtables.units import Qty

from cimbuilder.object_builder import BaseVoltageBuilder, FeederBuilder
from cimbuilder.substation_builder import BreakerAndHalfSubstationBuilder

### Load the IEEE 13 and IEEE 13 Assets feeders

Both feeder files load into one graph. The substation is built into the same graph, so that its switches
connect to the feeders' existing head nodes.

In [2]:
IEEE13 = "49AD8E07-3BF9-A4E2-CB8F-C3722F837B62"
IEEE13_ASSETS = "5B816B93-7A5F-B64C-8460-47C17D6E4B0F"
FEEDER_FILES = ["../../tests/test_models/IEEE13.xml", "../../tests/test_models/IEEE13_Assets.xml"]

network = FeederModel(
    container=cim.Feeder(mRID=IEEE13), connection=XMLFile(filename=FEEDER_FILES), distributed=False
)
container = network.container

Two small helpers: `sourcebus` finds a feeder's head node, the `ConnectivityNode` named `sourcebus` inside
that feeder, and `show` prints each switch with the nodes on its two terminals.

In [3]:
def sourcebus(network, feeder_mRID):
    """mRID of the feeder head node, the ConnectivityNode named sourcebus in the feeder"""
    for node in network.graph[cim.ConnectivityNode].values():
        node_container = node.ConnectivityNodeContainer
        if node.name == "sourcebus" and node_container is not None and node_container.mRID == feeder_mRID:
            return node.mRID
    raise ValueError(f"no sourcebus in feeder {feeder_mRID}")


def show(switches):
    """one line per switch: class, mRID, the nodes on its two terminals and normalOpen"""
    for switch in switches:
        node1, node2 = [terminal.ConnectivityNode for terminal in switch.Terminals]
        print(
            f"{type(switch).__name__:13} {switch.mRID:24} "
            f"{node1.name or node1.mRID} - {node2.name or node2.mRID}   normalOpen={switch.normalOpen}"
        )


print("IEEE 13 sourcebus:       ", sourcebus(network, IEEE13))
print("IEEE 13 Assets sourcebus:", sourcebus(network, IEEE13_ASSETS))

IEEE 13 sourcebus:        A8A25B50-3AE3-4A31-A18B-B3FA13397ED3
IEEE 13 Assets sourcebus: E14BD0DF-466D-487B-8786-0D46515D1F9B


### Create the Substation and its 115 kV VoltageLevel

References between objects are mRID strings, as in every other builder. The `BaseVoltage` is created first
so the VoltageLevel can refer to it.

In [4]:
BaseVoltageBuilder(network, container).create(
    mRID="bv_115kV", name="115 kV", nominalVoltage=Qty(115, "kV")
)

builder = BreakerAndHalfSubstationBuilder(network, container)
substation = builder.create(mRID="sub_breaker_and_half", name="sub_breaker_and_half")
substation, voltage_level = builder.add_voltage_level(
    substation, mRID="vl_115", name="115 kV", BaseVoltage="bv_115kV"
)

### Add the buses

`diameters` sets the number of diameters (default 2). Each diameter is three breaker bays, nine switches in
all.

In [5]:
voltage_level, busbars, diameters = builder.add_buses(voltage_level, diameters=2)
print([busbar.mRID for busbar in busbars])
show(diameters)

['vl_115_bus_1_bbs', 'vl_115_bus_2_bbs']
Disconnector  vl_115_dia_1_1_d1        vl_115_bus_1 - vl_115_dia_1_1_j1   normalOpen=False
Breaker       vl_115_dia_1_1_brk       vl_115_dia_1_1_j1 - vl_115_dia_1_1_j2   normalOpen=False
Disconnector  vl_115_dia_1_1_d2        vl_115_dia_1_1_j2 - vl_115_dia_1_side_1   normalOpen=False
Disconnector  vl_115_dia_1_2_d1        vl_115_dia_1_side_1 - vl_115_dia_1_2_j1   normalOpen=False
Breaker       vl_115_dia_1_2_brk       vl_115_dia_1_2_j1 - vl_115_dia_1_2_j2   normalOpen=False
Disconnector  vl_115_dia_1_2_d2        vl_115_dia_1_2_j2 - vl_115_dia_1_side_2   normalOpen=False
Disconnector  vl_115_dia_1_3_d1        vl_115_dia_1_side_2 - vl_115_dia_1_3_j1   normalOpen=False
Breaker       vl_115_dia_1_3_brk       vl_115_dia_1_3_j1 - vl_115_dia_1_3_j2   normalOpen=False
Disconnector  vl_115_dia_1_3_d2        vl_115_dia_1_3_j2 - vl_115_bus_2   normalOpen=False
Disconnector  vl_115_dia_2_1_d1        vl_115_bus_1 - vl_115_dia_2_1_j1   normalOpen=False
Breake

### Add the branches

`diameter` and `side` (1, next to bus 1, or 2, next to bus 2) pick the connection point. Each side takes
one branch.

In [6]:
voltage_level, switches = builder.add_branch(voltage_level, diameter=1, side=1, node="line_1")
show(switches)
voltage_level, switches = builder.add_branch(
    voltage_level, diameter=1, side=2, node=sourcebus(network, IEEE13)
)
show(switches)
voltage_level, switches = builder.add_branch(
    voltage_level, diameter=2, side=1, node=sourcebus(network, IEEE13_ASSETS)
)
show(switches)

Disconnector  vl_115_dia_1_side_1_d    vl_115_dia_1_side_1 - line_1   normalOpen=False
Disconnector  vl_115_dia_1_side_2_d    vl_115_dia_1_side_2 - sourcebus   normalOpen=False
Disconnector  vl_115_dia_2_side_1_d    vl_115_dia_2_side_1 - sourcebus   normalOpen=False


### Link the feeders to the substation

The switches connect the feeders electrically. The feeder containers link to the substation through
`Feeder.NormalEnergizingSubstation`:

In [7]:
feeder_builder = FeederBuilder(network, container)
for feeder_mRID in (IEEE13, IEEE13_ASSETS):
    feeder = network.find_by_mRID(feeder_mRID)
    feeder_builder.add_connectivity(feeder, NormalEnergizingSubstation=substation.mRID)

print([feeder.name for feeder in substation.NormalEnergizedFeeder])

['ieee13nodeckt', 'ieee13nodecktassets']


### Save the substation and feeders to CIM XML

In [8]:
os.makedirs("output", exist_ok=True)
utils.write_xml(network, "output/breaker_and_half.xml")

## Read the XML file back

Reload the file into a new model and check that the substation, its feeders and its switching equipment
all came back, along with the feeders' load.

In [9]:
roundtrip = FeederModel(
    container=cim.Feeder(mRID=IEEE13),
    connection=XMLFile(filename="output/breaker_and_half.xml"),
    distributed=False,
)
substation = roundtrip.find_by_mRID("sub_breaker_and_half")
voltage_level = roundtrip.find_by_mRID("vl_115")
print("feeders:", [feeder.name for feeder in substation.NormalEnergizedFeeder])
for cim_class in (cim.BusbarSection, cim.Breaker, cim.Disconnector):
    count = sum(isinstance(equipment, cim_class) for equipment in voltage_level.Equipments)
    print(f"{cim_class.__name__}: {count}")

feeders: ['ieee13nodeckt', 'ieee13nodecktassets']
BusbarSection: 2
Breaker: 6
Disconnector: 15


In [10]:
# CIM XML values are in SI base units (W); CIMUnit converts them for display
total_load = sum(float(load.p) for load in roundtrip.graph[cim.EnergyConsumer].values())
print(f"total load served by the substation: {cim.ActivePower(total_load).to('kW'):.1f} kW")

total load served by the substation: 6937.0 kW
